In [4]:
import sys
import json
import re
from pathlib import Path
from datetime import datetime

import fitz
from IPython.display import display, Markdown

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from config.settings import (
    PDF_PATH,
    TEXT_DIR,
    RAW_TEXT_PATH,
    CLEAN_TEXT_PATH,
    HEADER_FOOTER_PATTERNS,
    PAGE_NUMBER_PATTERN,
    OUTPUT_DIRS,
)

for d in OUTPUT_DIRS:
    d.mkdir(parents=True, exist_ok=True)

In [5]:
assert PDF_PATH.exists(), f"PDF not found: {PDF_PATH}"

doc = fitz.open(str(PDF_PATH))
raw_pages = []

for i in range(len(doc)):
    text = doc[i].get_text("text")
    words = text.split()
    raw_pages.append({
        "page": i + 1,
        "raw_text": text,
        "raw_char_count": len(text),
        "raw_word_count": len(words),
    })

doc.close()

RAW_TEXT_PATH.write_text(json.dumps(raw_pages, indent=2, ensure_ascii=False), encoding="utf-8")

display(Markdown(f"""
### Raw Text Extraction Complete

| Item | Value |
|------|-------|
| **Pages extracted** | {len(raw_pages)} |
| **Saved to** | `{RAW_TEXT_PATH.relative_to(project_root)}` |
""".strip()))

### Raw Text Extraction Complete

| Item | Value |
|------|-------|
| **Pages extracted** | 1314 |
| **Saved to** | `SCADA-DIP\data\text\pages_raw.json` |

In [6]:
_page_re = re.compile(PAGE_NUMBER_PATTERN, re.IGNORECASE)


def is_header_footer(line):
    stripped = line.strip()
    if not stripped:
        return False
    for pattern in HEADER_FOOTER_PATTERNS:
        if stripped == pattern:
            return True
    if _page_re.match(stripped):
        return True
    return False


def clean_page_text(raw_text):
    lines = raw_text.split("\n")
    kept = []
    removed = []

    for line in lines:
        if is_header_footer(line):
            removed.append(line.strip())
        else:
            kept.append(line)

    clean = "\n".join(kept)
    # Normalize excessive blank lines
    clean = re.sub(r"\n{3,}", "\n\n", clean)
    return clean.strip(), removed

In [7]:
clean_pages = []

for entry in raw_pages:
    clean_text, removed = clean_page_text(entry["raw_text"])
    clean_words = clean_text.split()

    clean_pages.append({
        "page": entry["page"],
        "raw_text": entry["raw_text"],
        "clean_text": clean_text,
        "raw_char_count": entry["raw_char_count"],
        "clean_char_count": len(clean_text),
        "raw_word_count": entry["raw_word_count"],
        "clean_word_count": len(clean_words),
        "removed_lines": removed,
        "has_text": len(clean_text.strip()) > 0,
    })

CLEAN_TEXT_PATH.write_text(json.dumps(clean_pages, indent=2, ensure_ascii=False), encoding="utf-8")

display(Markdown(f"""
### Cleaned Text Saved

| Item | Value |
|------|-------|
| **Pages** | {len(clean_pages)} |
| **Saved to** | `{CLEAN_TEXT_PATH.relative_to(project_root)}` |
""".strip()))

### Cleaned Text Saved

| Item | Value |
|------|-------|
| **Pages** | 1314 |
| **Saved to** | `SCADA-DIP\data\text\pages_clean.json` |

In [8]:
total_pages = len(clean_pages)
pages_with_text = sum(1 for p in clean_pages if p["has_text"])
pages_without_text = total_pages - pages_with_text

total_raw_chars = sum(p["raw_char_count"] for p in clean_pages)
total_clean_chars = sum(p["clean_char_count"] for p in clean_pages)
chars_removed = total_raw_chars - total_clean_chars
removal_pct = (chars_removed / total_raw_chars * 100) if total_raw_chars else 0

total_removed_lines = sum(len(p["removed_lines"]) for p in clean_pages)

# Count removed line types
from collections import Counter
removed_counter = Counter()
for p in clean_pages:
    for line in p["removed_lines"]:
        if _page_re.match(line):
            removed_counter["Page X of 1314"] += 1
        else:
            removed_counter[line] += 1

reason_rows = "\n".join(
    f"| `{pattern}` | {count} |"
    for pattern, count in removed_counter.most_common(10)
)

display(Markdown(f"""
### Cleaning Impact Summary

| Item | Value |
|------|-------|
| **Total pages** | {total_pages} |
| **Pages with text** | {pages_with_text} |
| **Pages without text after cleaning** | {pages_without_text} |
| **Total raw characters** | {total_raw_chars:,} |
| **Total clean characters** | {total_clean_chars:,} |
| **Characters removed** | {chars_removed:,} ({removal_pct:.1f}%) |
| **Total lines removed** | {total_removed_lines:,} |

#### Removed Line Breakdown (top 10)

| Pattern | Count |
|---------|-------|
{reason_rows}
""".strip()))

### Cleaning Impact Summary

| Item | Value |
|------|-------|
| **Total pages** | 1314 |
| **Pages with text** | 1313 |
| **Pages without text after cleaning** | 1 |
| **Total raw characters** | 1,818,556 |
| **Total clean characters** | 1,759,314 |
| **Characters removed** | 59,242 (3.3%) |
| **Total lines removed** | 3,932 |

#### Removed Line Breakdown (top 10)

| Pattern | Count |
|---------|-------|
| `System Guide` | 1311 |
| `7EN02-0440-03` | 1311 |
| `Page X of 1314` | 1310 |

In [9]:
sample_pages = [43, 50, 100, 500, 1000]
sample_pages = [p for p in sample_pages if p <= total_pages]

for pg in sample_pages:
    entry = clean_pages[pg - 1]
    removed_str = ", ".join(f"`{r}`" for r in entry["removed_lines"]) if entry["removed_lines"] else "None"
    preview = entry["clean_text"][:500] + ("..." if len(entry["clean_text"]) > 500 else "")

    display(Markdown(f"""
---
#### Page {pg} | Raw: {entry['raw_char_count']} chars | Clean: {entry['clean_char_count']} chars | Removed lines: {len(entry['removed_lines'])}

**Removed:** {removed_str}

**Clean text preview:**
```
{preview}
```
""".strip()))

---
#### Page 43 | Raw: 1299 chars | Clean: 1255 chars | Removed lines: 3

**Removed:** `System Guide`, `7EN02-0440-03`, `Page 43 of 1314`

**Clean text preview:**
```
• Support for McAfee Application Control software to help protect against zero day attacks.
• Windows Active Directory integration, role-based access control, and two-factor authen-
tication using YubiKey.
• Power SCADA Runtime user partitioning (8 levels of user privilege) and user event monitoring
(log in, log out, shutdown, control, etc.).
Recommended actions
You must take steps to help secure your system at every stage of the project life-cycle. The
following table lists the actions we recom...
```

---
#### Page 50 | Raw: 2094 chars | Clean: 2050 chars | Removed lines: 3

**Removed:** `System Guide`, `Page 50 of 1314`, `7EN02-0440-03`

**Clean text preview:**
```
Plan
• I/O point count is now tag based not address based. For example, two tags that use the same
PLC address will be counted twice. If two trend tags use the same variable tag, it will be coun-
ted once. The same applies to alarms.
• For the multi-process mode, each server component will accumulate its own point count. The
server component point count is the count added up from all server components. If two server
components use the same tags, say alarm and trend, the tags will be counted twic...
```

---
#### Page 100 | Raw: 1090 chars | Clean: 1045 chars | Removed lines: 3

**Removed:** `System Guide`, `Page 100 of 1314`, `7EN02-0440-03`

**Clean text preview:**
```
Plan
EcoStruxure Web Services (EWS)
EcoStruxure Web Services (EWS) for Power SCADA Operation shares real-time, historical, and
alarm data with EcoStruxure™Building Operations (EBO) and historical data with Power
Monitoring Expert (PME). Do not confuse this feature with the EWS Server that was released as a
part of PowerSCADA Expert/Vijeo Citect version 7.40 (which is for tag level process data).
EWS uses web-based HTTP protocol to transfer data. It enables two-way data transfers, which
allows th...
```

---
#### Page 500 | Raw: 829 chars | Clean: 784 chars | Removed lines: 3

**Removed:** `System Guide`, `Page 500 of 1314`, `7EN02-0440-03`

**Clean text preview:**
```
Configure
Snippet type
Syntax
Attributes Description
and password.
Optional Attributes for snippet
Type User credential PopUp
ShowTitleBar: Displays the title
bar in the target pane when set to
Yes.
Adding a diagram to the menu bar
You can add a diagram to the menu bar and then use it to navigate to diagrams.
This topic uses an example to demonstrate how to accomplish this.
To add a diagram to the menu bar:
1. Log in to PSO Web Applications (https//localhost/webhmI or ipaddress/webhmi).
The foll...
```

---
#### Page 1000 | Raw: 2058 chars | Clean: 2012 chars | Removed lines: 3

**Removed:** `System Guide`, `Page 1000 of 1314`, `7EN02-0440-03`

**Clean text preview:**
```
Reference
5. Click Initialize. Mapped Trend logs appear with associated timestamp data for each.
You should see a row for each pair selected in the Mappings tab. The Key is a long string
that represents the pair.
Now, the next time you run ETL, only data after the given timestamp is loaded.
6. Run the ETL job.
The Target Device value is the PME source under which the PSO data will be loaded.
7. (Optional) Verify the data transfer. See "Verifying PSO data transfer to PME" on page 1000
for details...
```

In [10]:
buckets = {"0 (no text)": 0, "1-500": 0, "501-1500": 0, "1501-3000": 0, "3000+": 0}

for p in clean_pages:
    c = p["clean_char_count"]
    if c == 0:
        buckets["0 (no text)"] += 1
    elif c <= 500:
        buckets["1-500"] += 1
    elif c <= 1500:
        buckets["501-1500"] += 1
    elif c <= 3000:
        buckets["1501-3000"] += 1
    else:
        buckets["3000+"] += 1

rows = "\n".join(f"| {k} | {v} |" for k, v in buckets.items())

display(Markdown(f"""
### Clean Text Length Distribution

| Characters | Pages |
|------------|-------|
{rows}
""".strip()))

### Clean Text Length Distribution

| Characters | Pages |
|------------|-------|
| 0 (no text) | 1 |
| 1-500 | 221 |
| 501-1500 | 513 |
| 1501-3000 | 577 |
| 3000+ | 2 |

In [ ]:
sample = json.loads(CLEAN_TEXT_PATH.read_text(encoding="utf-8"))[0]

sample_preview = {
    "page": sample["page"],
    "raw_text": sample["raw_text"][:200] + "...",
    "clean_text": sample["clean_text"][:200] + "...",
    "raw_char_count": sample["raw_char_count"],
    "clean_char_count": sample["clean_char_count"],
    "raw_word_count": sample["raw_word_count"],
    "clean_word_count": sample["clean_word_count"],
    "removed_lines": sample["removed_lines"],
    "has_text": sample["has_text"],
}

display(Markdown(f"""
## Sample Output Shape

```json
{json.dumps(sample_preview, indent=2, ensure_ascii=False)}
```
""".strip()))

## Sample Output Shape

```json
{
  "page": 1,
  "raw_text": "EcoStruxure™\nPower SCADA Operation 2020 with Advanced\nReporting and Dashboards\nSystem Guide\n7EN02-0440-03\n04/2022\n...",
  "clean_text": "EcoStruxure™\nPower SCADA Operation 2020 with Advanced\nReporting and Dashboards\n04/2022...",
  "raw_char_count": 114,
  "clean_char_count": 86,
  "raw_word_count": 14,
  "clean_word_count": 11,
  "removed_lines": [
    "System Guide",
    "7EN02-0440-03"
  ],
  "has_text": true
}
```


---

## Notebook 03 Complete — Text Extraction and Cleaning

| Deliverable | Path |
|-------------|------|
| Raw page text | `{RAW_TEXT_PATH.relative_to(project_root)}` |
| Clean page text | `{CLEAN_TEXT_PATH.relative_to(project_root)}` |

- {total_pages} pages extracted and cleaned.
- {total_removed_lines:,} header/footer lines removed ({removal_pct:.1f}% character reduction).
- {pages_with_text} pages contain usable text after cleaning.
- Output is ready for **TOC-aware chunking** in the next notebook.

**Next step:** `04_toc_aware_chunking.ipynb`